# Comparisons and uncertainty

11 November 2026 · Lecture 2

Run Python cells from the repository root. Prepare local data with `uv run python scripts/fetch_data.py`. Paths are relative to that root.

## About the data

`amazon_purchases.csv` is an extract of Berke and colleagues' Open e-commerce 1.0 dataset. US volunteers donated their Amazon purchase histories. Each row is a product-purchase record with a date, shipping state, category, unit price, quantity and calculated spending in USD. We retain 2018–2022 and seven named product categories. The extract omits participant identifiers and product titles. This convenience sample is not representative of all Amazon customers or total company revenue.

`event_study_results.csv` contains published estimates from Callaway and Sant'Anna's did-package real-data example: minimum-wage increases and log teen employment in a 500-county panel, 2003–2007. Each row is a year relative to the increase, with an estimate and published 95% simultaneous confidence-band bounds. We plot their supplied results rather than estimate a model. The authors describe this as an illustrative application, not a definitive policy evaluation.

Sources, licences, selection rules and limitations are in the [data documentation](../../data/README.md). Files are prepared by `scripts/fetch_data.py`; this notebook reads local data only.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Purchase prices: why an average can hide the distribution

Compare Boots and Sandals in the donated Amazon histories. Each observation is a
purchase record. Frequent shoppers contribute repeatedly, so this is not a sample
of independent customers. Inspect large values rather than deleting them automatically.

In [ ]:
# parse_dates reads the date column as dates, enabling .dt below.
purchases = pd.read_csv("data/raw/amazon_purchases.csv", parse_dates=["date"])
# isin keeps both footwear categories; copy() creates a separate working table.
footwear = purchases.loc[purchases["category"].isin(["Boots", "Sandals"])].copy()
# agg applies several summaries to unit prices within each category.
# count reports non-missing prices, not distinct shoppers.
print(footwear.groupby("category")["unit_price_usd"].agg(["count", "mean", "median", "min", "max"]))

## Histogram bins and box plots

Both histograms contain the same prices. Check which features survive a change in bins.
The box plot then compares the two categories on the same scale. Box-plot whiskers
describe observed variation, not uncertainty about the median or mean.

In [ ]:
# Make one row of two panels; sharex=True gives both the same x-axis limits.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
# strict=True checks that the panel and bin-count sequences have equal lengths.
for ax, bins in zip(axes, [15, 40], strict=True):
    # bins controls the number of intervals; edgecolor outlines their bars.
    ax.hist(footwear["unit_price_usd"], bins=bins, edgecolor="white")
    ax.set(title=f"{bins} bins", xlabel="Price per item (USD)", ylabel="Purchase records")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
# Compare both categories on one price scale, including outliers.
sns.boxplot(data=footwear, x="category", y="unit_price_usd", color="#9DBFCA", ax=ax)
# xlabel="" removes a redundant category-axis title.
ax.set(title="Boot and sandal prices in donated purchase histories", xlabel="", ylabel="Price per item (USD)")
# Put the source note near the figure's bottom-left, outside the axes.
fig.text(0.01, 0.01, "Berke et al., Open e-commerce 1.0. Selected categories, 2018–2022. No price outliers removed.", fontsize=8)
# Reserve the bottom 9% of the figure for that note.
fig.tight_layout(rect=(0, 0.09, 1, 1))
plt.show()

## Seasonal comparisons

Count purchase records by calendar month and category, pooled over 2018–2022.
A record may contain more than one item. This view asks about recorded purchases,
not revenue or distinct shoppers. Which footwear category peaks in colder months?

In [ ]:
# .dt.month extracts month numbers (1–12), pooling the years here.
# size() counts rows in each month/category group.
# unstack makes categories into columns; absent combinations become 0.
seasonal = footwear.groupby([footwear["date"].dt.month, "category"]).size().unstack(fill_value=0)
print(seasonal)
fig, ax = plt.subplots(figsize=(9, 4))
# Plot one line per category column; marker="o" adds a dot at each month.
seasonal.plot(ax=ax, marker="o")
ax.set(title="Footwear purchase records by calendar month", xlabel="Month number", ylabel="Records, 2018–2022 combined", xticks=range(1, 13))
fig.tight_layout()
plt.show()

## Published estimates and uncertainty

Now switch from variation in purchases to uncertainty about statistical estimates.
Callaway and Sant'Anna's real-data example studies minimum-wage increases and log
teen employment in 500 US counties, 2003–2007. We use their published `mw.dyn` output,
without covariates, with never-treated counties as the comparison group.

The horizontal coordinate is years since a county's first minimum-wage increase.
The vertical coordinate is an estimated difference in log employment. The supplied
bounds form a **95% simultaneous confidence band**, not 95% of individual observations.
Use these bounds directly; do not replace them with estimate ± 1.96 × standard error.

Before treatment, the published default compares successive pre-treatment periods.
In particular, year -1 is an estimated pre-treatment contrast, **not a zero normalisation**.
Post-treatment comparisons use the last pre-treatment year. The cohorts contributing
to each event time vary. The authors caution that this example is illustrative, not a
definitive evaluation of minimum-wage policy.

In [ ]:
# These are supplied estimates and confidence bounds, not raw observations.
event = pd.read_csv("data/processed/event_study_results.csv")
print(event)
# errorbar needs distances from each estimate, not the endpoints themselves.
# The first list entry is downward distances; the second is upward distances.
errors = [event["estimate"] - event["ci_low"], event["ci_high"] - event["estimate"]]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
# yerr supplies the distances; fmt="o" draws dots; capsize adds end caps.
ax.errorbar(event["event_year"], event["estimate"], yerr=errors, fmt="o", capsize=4)
# Zero is the reference for no estimated change in log employment.
ax.axhline(0, color="grey", linewidth=1)
# A dashed vertical line separates pre-treatment years from year 0 onward.
ax.axvline(-0.5, color="grey", linestyle="--", linewidth=1)
# Place a tick at each supplied event year.
ax.set(title="Minimum-wage increases and teen employment", xlabel="Years relative to first increase (0 = first year)", ylabel="Estimated change in log employment", xticks=event["event_year"])
# Add the source and interval definition below the plot.
fig.text(0.01, 0.02, "Callaway & Sant'Anna, did real-data example. Published 95% simultaneous bands; illustrative application.", fontsize=8)
fig.tight_layout(rect=(0, 0.10, 1, 1))
# Save at 150 dots per inch, including the note in the tight image bounds.
fig.savefig("figures/02_event_study.png", dpi=150, bbox_inches="tight")
plt.show()

Which estimates have bands spanning zero? That does not prove a zero effect. Which
periods are relatively uncertain? What would disappear if we showed only the points?
The chart does not itself establish the assumptions needed for causal interpretation.

[Published example](https://bcallaway11.github.io/did/articles/did-basics.html#an-example-with-real-data)
and [source details](../../data/README.md).

Open [the exercises](exercises.ipynb), then briefly exchange project questions and an example figure.

Course AI use and disclosure requirements are in [the policy](../../docs/ai_policy.md). If you use assistance, verify the code, numbers and interpretation and disclose it as required.